In [2]:
from pathlib import Path

import pandas as pd

from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.tools import tool

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from langchain_groq import ChatGroq

from langgraph.prebuilt import create_react_agent

In [3]:
load_dotenv()

print("Environment variables loaded.")

Environment variables loaded.


In [4]:
data_path = Path("../data/structured")

employees = pd.read_csv(
    data_path / "employees.csv"
)

attendance = pd.read_csv(
    data_path / "attendance.csv"
)

leave = pd.read_csv(
    data_path / "leave.csv"
)

holidays = pd.read_csv(
    data_path / "holidays.csv"
)

attendance["date"] = pd.to_datetime(
    attendance["date"]
)

leave["start_date"] = pd.to_datetime(
    leave["start_date"]
)

leave["end_date"] = pd.to_datetime(
    leave["end_date"]
)

holidays["date"] = pd.to_datetime(
    holidays["date"]
)

print("Structured data loaded.")

Structured data loaded.


In [5]:
kb_path = Path("../data/knowledge_base")

documents = []

for file in kb_path.glob("*.md"):

    text = file.read_text(
        encoding="utf-8"
    )

    documents.append(
        Document(
            page_content=text,
            metadata={
                "source": file.name
            }
        )
    )

print(
    f"Loaded {len(documents)} policy documents."
)

Loaded 6 policy documents.


In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=150,
    separators=[
        "\n\n",
        "\n",
        ".",
        " "
    ]
)

chunks = splitter.split_documents(
    documents
)

print(
    f"Created {len(chunks)} policy chunks."
)

Created 8 policy chunks.


In [7]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5507.99it/s]


Embedding model loaded.


In [8]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="nexatech_agent_policies"
)

print(
    "Vectors stored:",
    vector_store._collection.count()
)

Vectors stored: 8


In [9]:
retriever = vector_store.as_retriever(
    search_kwargs={
        "k": 4
    }
)

print("Policy retriever ready.")

Policy retriever ready.


In [10]:
@tool
def search_company_policy(
    question: str
) -> str:
    """
    Search NexaTech company policies.

    Use this tool when the user asks about company rules,
    policies, requirements, working-from-home rules,
    attendance requirements, overtime, sick leave,
    vacation policies, or other workplace policies.

    Do NOT use this tool for questions asking about
    an employee's actual attendance records.
    """

    retrieved_docs = retriever.invoke(
        question
    )

    if not retrieved_docs:
        return (
            "No relevant company policy "
            "was found."
        )

    results = []

    for doc in retrieved_docs:

        results.append(
            f"Source: {doc.metadata['source']}\n"
            f"{doc.page_content}"
        )

    return "\n\n---\n\n".join(results)

In [11]:
@tool
def get_employee(
    employee_id: str
) -> str:
    """
    Get information about an employee.

    Use this when you need employee details such as
    name, department, role, employment type,
    weekly working hours, vacation entitlement,
    or office location.
    """

    result = employees[
        employees["employee_id"] == employee_id
    ]

    if result.empty:
        return (
            f"No employee found with ID "
            f"{employee_id}."
        )

    employee = result.iloc[0]

    return (
        f"Employee ID: {employee['employee_id']}\n"
        f"Name: {employee['name']}\n"
        f"Department: {employee['department']}\n"
        f"Role: {employee['role']}\n"
        f"Employment type: {employee['employment_type']}\n"
        f"Weekly hours: {employee['weekly_hours']}\n"
        f"Vacation entitlement: {employee['vacation_days']} days\n"
        f"Office location: {employee['office_location']}"
    )

In [12]:
@tool
def get_attendance_summary(
    employee_id: str,
    start_date: str,
    end_date: str
) -> str:
    """
    Get actual attendance statistics for an employee
    during a specified date range.

    Returns office days, home-office days,
    business-trip days, sick days, leave days,
    and missing attendance records.
    """

    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)

    records = attendance[
        (attendance["employee_id"] == employee_id) &
        (attendance["date"] >= start) &
        (attendance["date"] <= end)
    ]

    if records.empty:
        return (
            f"No attendance records found for "
            f"{employee_id}."
        )

    office_days = (
        records["location"] == "office"
    ).sum()

    home_days = (
        records["location"] == "home"
    ).sum()

    business_trip_days = (
        records["location"] == "business_trip"
    ).sum()

    sick_days = (
        records["status"] == "sick"
    ).sum()

    leave_days = (
        records["status"] == "leave"
    ).sum()

    missing_records = records[
        records["status"].isin(
            ["missing", "missing_checkout"]
        )
    ].shape[0]

    return (
        f"Employee: {employee_id}\n"
        f"Period: {start_date} to {end_date}\n"
        f"Office days: {office_days}\n"
        f"Home-office days: {home_days}\n"
        f"Business-trip days: {business_trip_days}\n"
        f"Sick days: {sick_days}\n"
        f"Leave days: {leave_days}\n"
        f"Missing attendance records: {missing_records}"
    )

In [13]:
@tool
def calculate_working_hours(
    employee_id: str,
    start_date: str,
    end_date: str
) -> str:
    """
    Calculate total recorded working hours
    for an employee during a date range.
    """

    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)

    records = attendance[
        (attendance["employee_id"] == employee_id) &
        (attendance["date"] >= start) &
        (attendance["date"] <= end)
    ].copy()

    records = records[
        records["check_in"].notna() &
        records["check_out"].notna() &
        (records["check_in"] != "") &
        (records["check_out"] != "")
    ]

    if records.empty:
        return (
            f"No complete attendance records "
            f"found for {employee_id}."
        )

    records["check_in_time"] = pd.to_datetime(
        records["check_in"],
        format="%H:%M"
    )

    records["check_out_time"] = pd.to_datetime(
        records["check_out"],
        format="%H:%M"
    )

    records["hours"] = (
        records["check_out_time"]
        - records["check_in_time"]
    ).dt.total_seconds() / 3600

    total_hours = records["hours"].sum()

    return (
        f"Employee: {employee_id}\n"
        f"Period: {start_date} to {end_date}\n"
        f"Total recorded working hours: "
        f"{total_hours:.2f}"
    )

In [14]:
@tool
def get_leave_balance(
    employee_id: str
) -> str:
    """
    Get an employee's vacation entitlement,
    approved vacation used, and remaining vacation.
    """

    employee_result = employees[
        employees["employee_id"] == employee_id
    ]

    if employee_result.empty:
        return (
            f"No employee found with ID "
            f"{employee_id}."
        )

    employee = employee_result.iloc[0]

    entitlement = int(
        employee["vacation_days"]
    )

    approved_vacation = leave[
        (leave["employee_id"] == employee_id) &
        (leave["type"] == "vacation") &
        (leave["status"] == "approved")
    ]

    used = int(
        approved_vacation["days"].sum()
    )

    remaining = entitlement - used

    return (
        f"Employee: {employee_id}\n"
        f"Vacation entitlement: {entitlement} days\n"
        f"Approved vacation used: {used} days\n"
        f"Remaining vacation: {remaining} days"
    )

In [15]:
@tool
def find_missing_attendance(
    employee_id: str,
    start_date: str,
    end_date: str
) -> str:
    """
    Find dates where an employee has missing
    attendance or missing check-out.
    """

    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)

    records = attendance[
        (attendance["employee_id"] == employee_id) &
        (attendance["date"] >= start) &
        (attendance["date"] <= end)
    ]

    missing = records[
        records["status"].isin(
            ["missing", "missing_checkout"]
        )
    ]

    if missing.empty:
        return (
            f"No missing attendance records found "
            f"for {employee_id}."
        )

    results = []

    for _, row in missing.iterrows():

        results.append(
            f"{row['date'].date()} - "
            f"{row['status']}"
        )

    return (
        f"Missing attendance for {employee_id}:\n"
        + "\n".join(results)
    )

In [16]:
tools = [
    search_company_policy,
    get_employee,
    get_attendance_summary,
    calculate_working_hours,
    find_missing_attendance,
    get_leave_balance
]

print("Available agent tools:\n")

for tool in tools:
    print("-", tool.name)

Available agent tools:

- search_company_policy
- get_employee
- get_attendance_summary
- calculate_working_hours
- find_missing_attendance
- get_leave_balance


In [17]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    reasoning_format="parsed"
)

print("LLM ready.")

LLM ready.


In [18]:
agent = create_react_agent(
    model=llm,
    tools=tools
)

print("Agent created.")

Agent created.


/var/folders/kb/3rhgzw5d6md74w5kr02cc4cm0000gn/T/ipykernel_1930/1727893791.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [19]:
question = """
How many days per week can employees normally
work from home?
"""

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": question
            }
        ]
    }
)

print(
    response["messages"][-1].content
)

Employees may normally work remotely for **up to two regular working days per week**. This is the standard limit for full‑time staff whose roles permit remote work, unless a manager grants a temporary exception for special circumstances.


In [20]:
question = """
How many days did E0001 work from home
in August 2026?
"""

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": question
            }
        ]
    }
)

print(
    response["messages"][-1].content
)

E0001 worked from home **5 days** during August 2026.


In [21]:
question = """
Did E0001 comply with the company's
remote-work policy in August 2026?
"""

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": question
            }
        ]
    }
)

print(
    response["messages"][-1].content
)

**Short answer:** Yes, based on the attendance data we have, E0001 appears to have complied with NexaTech’s remote‑work policy for August 2026.

### How the conclusion was reached

| Policy requirement | What the policy says | What the August 2026 data shows |
|--------------------|----------------------|---------------------------------|
| **Maximum remote days** | ≤ 2 remote (home‑office) days per regular work week. | 5 home‑office days in the whole month. Even if all 5 fell in the same week (the worst‑case scenario), that would still be ≤ 2 days per *regular* week because a week can contain at most 5 work days. In practice the 5 days are spread over the four weeks of August, giving an average of ≈ 1.25 remote days per week – well within the limit. |
| **Minimum office attendance** | ≥ 3 office days per week for hybrid employees. | 15 office days in August. August 2026 has 4 full work weeks (Mon‑Fri) plus a partial week at the start/end. 3 office days × 4 weeks = 12 required days; E0